# Environmental Solutions

:::{admonition} Reference solutions
:class: note
Worked solutions for the real Argo exercise in [1.3-numpy-environmental-exercises.ipynb](1.3-numpy-environmental-exercises.ipynb). Exercise 1 needs the `gsw` library, which is part of the book's environment.
:::

## Exercise 1: Ocean float profiles

```{figure} _static/argo-float-cycle.png
---
name: argo-float-cycle
width: 100%
alt: A diagram of one Argo float cycle, numbered one to seven, from deployment through drift, dive, ascent and satellite transmission
---
One cycle of an Argo float: deployment, descent to a drifting depth of about 1000 m, ten days of
drift, a dive to the profiling depth, then the ascent during which the measurements are taken, and
transmission to satellite at the surface before the next cycle begins. The panel on the right shows
what one ascent produces — the temperature and salinity profiles you are about to work with.
Illustration &copy; Thomas Haessig, from
<a href="https://www.euro-argo.eu/Outreach/Educational-material/Discover-Argo-floats-for-kids/How-do-the-Argo-floats-fulfill-their-mission-in-the-Ocean">Euro-Argo ERIC's educational material</a>.
```

[Argo](https://argo.ucsd.edu/) floats are autonomous instruments that drift with ocean currents,
periodically diving and surfacing while recording temperature, salinity, and pressure. Each dive
produces one *profile*: a set of measurements at many depth *levels*, with a single latitude,
longitude, and date attached.

This exercise uses real Argo data. Steps 1 to 8 use only what 1.1 to 1.3 cover: loading
arrays, inspecting shapes, rebuilding an axis, vectorised arithmetic, reductions along an axis,
boolean masks, and NaN-aware statistics. Steps 9 to 11 then plot what you computed.

:::{admonition} Beyond this subchapter
:class: note
Steps 9 to 11 use `matplotlib`, which gets its own treatment in 1.4. Import it once with
`import matplotlib.pyplot as plt`; after that you need only the functions below, and each is
named again where it is wanted:

- `plt.figure()` — start a new, empty figure; without it, every plot in a cell lands on the
  same axes
- [`plt.plot`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.plot.html) — a line, or
  one line per column when handed a 2D array
- [`plt.errorbar`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.errorbar.html) — a
  line with error bars
- [`plt.scatter`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.scatter.html) — points
- `plt.xlabel`, `plt.ylabel`, `plt.title` — each takes a string
- `plt.show()` — draw the current figure

An array of numbers with no picture attached is hard to sanity-check, which is why these three
steps sit here rather than waiting for 1.4: the profiles are the fastest way to see whether Steps 4
to 6 produced something physical.
:::

In [ ]:
# Pre-supplied: download and unzip the Argo data files.
import pooch

files = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/argo_float_data.zip",
    known_hash="sha256:dd913ba3450241bcd5d5697187ea68d398bdd7fd563a04cc74386834749c25f5",
    processor=pooch.Unzip(),
    path=pooch.os_cache("mlees"),
)
for f in sorted(files):
    print(f)

**Step 1.** Load each `.npy` file into a numpy array with `np.load`. Name them `T` (temperature,
°C), `S` (practical salinity, unitless), `P` (pressure, dbar), `date`, `lat`, `lon`, and `level` (depth level).

Read the filenames printed above to work out which file is which.

In [ ]:
import numpy as np
from pathlib import Path

data_dir = Path(files[0]).parent           # every file was unzipped into this one folder

T = np.load(data_dir / "T.npy")            # temperature, °C
S = np.load(data_dir / "S.npy")            # practical salinity, unitless
P = np.load(data_dir / "P.npy")            # pressure, dbar
date = np.load(data_dir / "date.npy")      # one date per profile
lat = np.load(data_dir / "lat.npy")
lon = np.load(data_dir / "lon.npy")
level = np.load(data_dir / "levels.npy")   # depth level index

**Step 2.** Print the `shape` of every array you loaded. Then answer in a comment: which
dimension do `T`, `S`, and `P` share with `level`, and which do they share with `lat`, `lon`,
and `date`?

In [ ]:
for name, arr in (("T", T), ("S", S), ("P", P),
                  ("date", date), ("lat", lat), ("lon", lon), ("level", level)):
    print(name, arr.shape)

# T, S and P are (78, 75). Their first axis, of length 78, is the one they share with
# level; their second axis, of length 75, is the one they share with date, lat and lon.
# So axis 0 runs down through depth and axis 1 runs across profiles.

**Step 3.** The `level` array is a regular sequence. Recreate it twice — once with `np.arange`
and once with `np.linspace` — and confirm each matches the original using `np.allclose`.

Read `level[0]`, `level[-1]`, and `level.size` first to work out the arguments.

In [ ]:
print(level[0], level[-1], level.size)

level_arange = np.arange(level[0], level[-1] + 1)          # stop is excluded
level_linspace = np.linspace(level[0], level[-1], level.size)   # both ends included

print(np.allclose(level_arange, level))
print(np.allclose(level_linspace, level))

**Step 4.** Compute the density of seawater relative to pure water, in kg m⁻³:

$$\rho - \rho_{\text{pure water}} = a\,S + b\,\Theta + c\,\Theta^{2}$$

where $S$ is salinity and $\Theta$ is the *conservative* temperature, computed from
temperature, salinity, and pressure by the `gsw` library. The constants are

```python
a = 7.718e-1
b = -8.44e-2
c = -4.561e-3
```

The cell below imports the function. Compute `conservative_temp` and `relative_density`, print
the shape of your result, and check it matches `T`.

Both `CT_from_t` and the formula expect *absolute* salinity, in g/kg, while Argo reports
*practical* salinity, which has no unit. Absolute salinity is about 0.5 % larger; for these
profiles, using practical salinity in its place lowers every relative density by about
0.13 kg m⁻³, which is small enough to ignore here. `gsw.SA_from_SP` does the exact conversion
when it matters.

Source: Roquet et al. (2015), *J. Phys. Oceanogr.* 45, 2564–2579, and the
[TEOS-10 GSW toolbox](https://www.teos-10.org/).

In [ ]:
# Pre-supplied: the conservative-temperature function from the gsw library
from gsw import CT_from_t

In [ ]:
a = 7.718e-1
b = -8.44e-2
c = -4.561e-3

conservative_temp = CT_from_t(S, T, P)          # salinity, temperature, pressure
# S is practical salinity standing in for absolute salinity: about 0.13 kg m-3 too light
relative_density = a * S + b * conservative_temp + c * conservative_temp**2

print(relative_density.shape, T.shape)
print(relative_density.shape == T.shape)

**Step 5.** Compute the mean and the standard deviation of `T`, `S`, `P`, and
`relative_density` **at each depth**, averaging across all the profiles.

Check your result has the same shape as `level` — if it does not, you reduced along the wrong
axis.

In [ ]:
mean_T = T.mean(axis=1)
mean_S = S.mean(axis=1)
mean_P = P.mean(axis=1)
mean_density = relative_density.mean(axis=1)

std_T = T.std(axis=1)
std_S = S.std(axis=1)
std_P = P.std(axis=1)
std_density = relative_density.std(axis=1)

print(mean_T.shape, level.shape)
print("first three mean temperatures:", mean_T[:3])

**Step 6.** Look at the mean you just computed. Many entries are `nan`, because Argo profiles
contain missing values and an ordinary mean propagates them.

1. Count how many values in `T` are missing, using `np.isnan` and a sum over the boolean mask.
2. Express that as a percentage of all values in `T`.
3. Recompute the mean and standard deviation with `np.nanmean` and `np.nanstd`, and confirm no
   `nan` remains.

In [ ]:
n_missing = np.isnan(T).sum()
print("missing values in T:", n_missing, "of", T.size)
print(f"that is {100.0 * n_missing / T.size:.1f} % of the array")

mean_T = np.nanmean(T, axis=1)
mean_S = np.nanmean(S, axis=1)
mean_P = np.nanmean(P, axis=1)
mean_density = np.nanmean(relative_density, axis=1)

std_T = np.nanstd(T, axis=1)
std_S = np.nanstd(S, axis=1)
std_P = np.nanstd(P, axis=1)
std_density = np.nanstd(relative_density, axis=1)

print("nan left in the mean temperature:", np.isnan(mean_T).sum())   # 0

**Step 7.** Using `np.where`, build an array that labels each depth `"warm"` where the mean
temperature there is above 10 °C and `"cold"` otherwise. Print the first ten labels alongside
the first ten depths.

Then, using a boolean mask, print the depths at which the mean temperature is below 5 °C.

In [ ]:
labels = np.where(mean_T > 10.0, "warm", "cold")
print(labels[:10])
print(level[:10])

cold_mask = mean_T < 5.0
print("depth levels colder than 5 °C:", level[cold_mask])

**Step 8.** Assemble a summary array with `np.vstack`: one row of depths, one of mean
temperature, one of mean salinity, and one of mean relative density — four rows in total.

Print its shape, save it to `_files/argo_summary.npy`, load it back, and confirm with `np.allclose`
that the round trip changed nothing.

In [ ]:
summary = np.vstack([level, mean_T, mean_S, mean_density])
print(summary.shape)

Path("_files").mkdir(exist_ok=True)
np.save("_files/argo_summary.npy", summary)
reloaded = np.load("_files/argo_summary.npy")
print(np.allclose(summary, reloaded))

**Step 9.** Plot each of `T`, `S`, `P` and `relative_density` against depth — four figures, one
per variable.

Each array is `(level, profile)`, so handing the whole array to `plt.plot` draws one line per
profile. Put the variable on the *x*-axis and `level` on the *y*-axis, which is the oceanographic
convention: depth runs down the page, and a profile is read the way the float measured it.

It will look messy, because 75 profiles are drawn on top of each other.

```{figure} _static/1.3-target-argo-profiles.png
---
name: target-argo-profiles
width: 500px
alt: Many overlapping salinity profiles plotted against depth level, forming a dense band that shifts towards fresher water with depth
---
The salinity panel, as a check on yours. Label the axes and give each figure a title.
```

In [ ]:
import matplotlib.pyplot as plt

for values, label in [
    (T, "Temperature (in degC)"),
    (S, "Salinity (in g/kg)"),
    (P, "Pressure (in dbar)"),
    (relative_density, "Relative density (in kg/m3)"),
]:
    plt.figure()
    plt.plot(values, level)      # values is (level, profile): one line per column
    plt.xlabel(label)
    plt.ylabel("Depth level")
    plt.title(f"{label.split(' (')[0]} profiles at different locations")
    plt.show()

**Step 10.** Now plot only the *mean* profile of each variable, with the standard deviation as
error bars.

Use the NaN-aware mean and standard deviation from Step 6, and draw them with `plt.errorbar`.
Because the variable is on the *x*-axis, the spread is a *horizontal* error bar — the keyword is
`xerr`, not `yerr`.

```{figure} _static/1.3-target-argo-mean-profiles.png
---
name: target-argo-mean-profiles
width: 500px
alt: A single mean salinity profile against depth level with horizontal error bars at each level
---
The salinity panel again, reduced from 75 profiles to one line and its spread.
```

In a comment, say what the width of the error bars tells you about how much the profiles disagree
near the surface compared with at depth.

In [ ]:
for mean_values, std_values, label in [
    (mean_T, std_T, "Temperature (in degC)"),
    (mean_S, std_S, "Salinity (in g/kg)"),
    (mean_P, std_P, "Pressure (in dbar)"),
    (mean_density, std_density, "Relative density (in kg/m3)"),
]:
    plt.figure()
    # the variable is on the x-axis, so the spread is a horizontal bar: xerr, not yerr
    plt.errorbar(mean_values, level, xerr=std_values)
    plt.xlabel(f"{label} including simple error bars")
    plt.ylabel("Depth level")
    plt.title(f"Mean {label.split(' (')[0].lower()} profile")
    plt.show()

# For temperature, salinity and density the bars are widest near the surface and narrow
# with depth. These 75 profiles span two years, and the upper ocean is where the seasons and
# the float's changing position make the most difference; the deep water is nearly the same
# in every profile. Pressure is the exception: its spread stays below 1.6 dbar at every level
# and does not shrink with depth, because each level is recorded at almost the same pressure
# in every profile.

**Step 11.** Scatter the profiles' longitudes against their latitudes.

One point per profile. Label both axes and give it a title, and set the marker size with `s=` so
the points are readable. Then print the first three entries of `date` and the last one.

In a comment, say what the dates and the shape of the cloud tell you about how this set of
profiles was collected, and by how many floats.

In [ ]:
plt.figure()
plt.scatter(lon, lat, s=20)
plt.xlabel("Longitude (in degrees east)")
plt.ylabel("Latitude (in degrees north)")
plt.title("Where each profile was taken")
plt.show()

print(date[:3])
print(date[-1])

# The dates are ten days apart and run from July 2012 to July 2014: 75 profiles on a steady
# ten-day cycle come from one float, not several. The points are that float's positions over
# the two years, spread over about 7 degrees of latitude (40-47 °N) and 8 of longitude
# (31-39 °W) in the North Atlantic. Because every profile comes from this one region, the deep
# parts of the mean profiles above agree so closely.